# Simulate table updates
Simulates a recurring pipeline load by appending sample orders to daily_orders. Run this notebook when fresh source data is needed for freshness validation.

In [0]:
%sql
-- Create the table on the first run only, then append rows on every run
create table if not exists validation_sweep.dev.daily_orders as
select * from samples.tpch_sf1.orders where o_orderkey <= 1000;

insert into validation_sweep.dev.daily_orders
select * from samples.tpch_sf1.orders limit 100;

In [0]:
# Verify the latest qualifying Delta data-change operation
from pyspark.sql import functions as F

data_change_operations = [
    "WRITE",
    "CREATE TABLE AS SELECT",
    "MERGE",
    "UPDATE",
    "DELETE",
    "COPY INTO",
    "STREAMING UPDATE",
]

latest = (
    spark.sql("describe history validation_sweep.dev.daily_orders")
    .filter(F.col("operation").isin(data_change_operations))
    .orderBy(F.col("timestamp").desc())
    .first()
)

if latest:
    print(f"Last data change: version {latest['version']}, "
          f"{latest['operation']} at {latest['timestamp']} UTC")
else:
    print("No identifiable data-changing operation")

In [0]:
%sql
-- Verify the latest Delta operation
describe history validation_sweep.dev.daily_orders;